In [1]:
# ── Colab / local setup ─────────────────────────────────────────────────────
# This cell installs magnowire from GitHub when running in Colab.
# Locally (if you cloned the repo) it does nothing.
import importlib, subprocess, sys

def _in_colab():
    try: import google.colab; return True
    except ImportError: return False

if _in_colab():
    # Replace with your GitHub username
    GITHUB_USER = "CostiD"   # <── change this
    REPO        = "magnowire"
    BRANCH      = "main"

    print(f"Colab detected — installing {GITHUB_USER}/{REPO} ...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        f"git+https://github.com/{GITHUB_USER}/{REPO}.git@{BRANCH}"
    ])
    print("Done. Runtime does NOT need to restart.")
else:
    # Running locally — add repo root to path
    import os
    sys.path.insert(0, os.path.join(os.getcwd(), ".."))
    print("Local run — using repo from:", os.path.abspath(".."))


Colab detected — installing CostiD/magnowire ...


CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', '--quiet', 'git+https://github.com/CostiD/magnowire.git@main']' returned non-zero exit status 2.

> **Colab tip**: For GPU acceleration go to
> *Runtime → Change runtime type → T4 GPU*
> before running. The solver auto-detects CuPy on Colab's CUDA environment.


# Example 01 — Single Co Nanowire

**Goal**: Validate the solver against the Stoner-Wohlfarth (SW) model,
then sweep wire diameter to observe the transition from coherent reversal
(thin wire, SW regime) to curling reversal (thick wire).

## Expected physics

For a cylindrical nanowire, field along the wire axis (x):

| Regime | Diameter | Mechanism | Predicted Hc |
|---|---|---|---|
| Stoner-Wohlfarth | d << l_ex | Coherent rotation | mu0 Hc ~ mu0 Ha = 643 mT |
| Curling | d >> l_ex | Vortex nucleation | Hc drops with d |

For Co: l_ex = 4.9 nm, mu0 Ha = 2 Ku/Ms = 643 mT.


---
## 0. Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import magnowire as mw
from magnowire import materials, geometry, solver, hysteresis, analysis
from magnowire._backend import to_xp, to_np, xp
from magnowire.constants import MU0

print(f"magnowire {mw.__version__}  |  backend: {mw.BACKEND}")
print()
materials.list_materials()


---
## 1. Validation tests

Before any simulation, verify the demag kernels pass all 5 tests.


In [ ]:
from magnowire.tests.test_demag import run_all
all_ok = run_all()
assert all_ok, "Some tests failed — check demag implementation"


---
## 2. Cobalt material parameters


In [ ]:
Co = materials.Co
print(Co)
print()
print(f"  Exchange length  l_ex   = {Co.l_ex*1e9:.2f} nm")
print(f"  Anisotropy field mu0 Ha = {Co.mu0_Ha*1e3:.1f} mT")
print(f"  Saturation Js          = {Co.mu0_Ms*1e3:.0f} mT")
print()
print("Stoner-Wohlfarth prediction (d << l_ex):")
print(f"  mu0 Hc ~ {Co.mu0_Ha*1e3:.0f} mT,  Mr/Ms = 1,  squareness = 1")


---
## 3. Geometry — d=10 nm, L=200 nm

Cell = 2 nm < l_ex (4.9 nm) — well resolved.


In [ ]:
geom = geometry.nanowire(
    diameter_nm = 10.,
    length_nm   = 200.,
    cell_nm     = 2.,
    axis        = "x",
    pad_cells   = 2,
    initial     = "uniform+x",
)
print(geom)
print(f"  cell / l_ex = {geom.dx / Co.l_ex:.2f}  (should be < 1)")

# Visualise mask
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.imshow(geom.mask[:, :, geom.nz//2].T, cmap="Blues",
          origin="lower", aspect="auto",
          extent=[0, geom.nx*geom.dx*1e9, 0, geom.ny*geom.dy*1e9])
ax.set_xlabel("x (wire axis) [nm]"); ax.set_ylabel("y [nm]")
ax.set_title("xy slice (z = centre)")

ax2 = axes[1]
ax2.imshow(geom.mask[geom.nx//2, :, :].T, cmap="Blues",
           origin="lower", aspect="equal",
           extent=[0, geom.ny*geom.dy*1e9, 0, geom.nz*geom.dz*1e9])
ax2.set_xlabel("y [nm]"); ax2.set_ylabel("z [nm]")
ax2.set_title("yz cross-section")

plt.suptitle(f"Geometry: {geom.name}", fontweight="bold")
plt.tight_layout(); plt.show()


---
## 4. Hysteresis loop — d=10 nm (SW regime)

Field along x, B_max=1.2 T > mu0 Ha = 0.64 T.


In [ ]:
sim = solver.MicromagSolver(
    geom     = geom,
    material = Co,
    pbc      = False,
    alpha    = 1.0,   # max damping for quasi-static
)

loop_10nm = hysteresis.hysteresis_loop(
    solver      = sim,
    B_max       = 1.2,
    n_field     = 21,
    field_axis  = 0,
    t_relax_ps  = 500.,
    dt          = 5e-13,
)


---
## 5. Figures of merit vs Stoner-Wohlfarth


In [ ]:
metrics_10nm = analysis.extract_metrics(loop_10nm, Ms_material=Co.Ms)
print("=" * 50)
print("d = 10 nm Co nanowire")
print("=" * 50)
print(metrics_10nm)
print()
print(f"  SW prediction:  mu0 Hc = {Co.mu0_Ha*1e3:.0f} mT")
print(f"  Simulation:     mu0 Hc = {metrics_10nm.Hc*1e3:.1f} mT")
print(f"  Relative diff:  {abs(metrics_10nm.Hc - Co.mu0_Ha)/Co.mu0_Ha*100:.1f}%")

fig, ax = plt.subplots(figsize=(7, 5))
analysis.plot_loop(loop_10nm, metrics_10nm,
                   title="Co nanowire d=10 nm, L=200 nm", ax=ax)
ax.axvline( Co.mu0_Ha*1e3, color="orange", lw=1.5, ls="--", label="SW Hc")
ax.axvline(-Co.mu0_Ha*1e3, color="orange", lw=1.5, ls="--")
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()


---
## 6. Diameter sweep — coherent vs curling transition

Sweep d = 4 to 40 nm. Expected: Hc and squareness drop as d increases.


In [ ]:
diameters_nm = [4., 8., 12., 20., 30., 40.]
sweep_results = {}   # diameter -> (loop, metrics)

for d_nm in diameters_nm:
    print(f"\n>>> d = {d_nm:.0f} nm <<<")
    g = geometry.nanowire(diameter_nm=d_nm, length_nm=200., cell_nm=2.)
    s = solver.MicromagSolver(g, Co, pbc=False, alpha=1.0, verbose=False)
    loop = hysteresis.hysteresis_loop(
        s, B_max=1.4, n_field=17, field_axis=0,
        t_relax_ps=400., dt=5e-13, verbose=True,
    )
    m = analysis.extract_metrics(loop, Ms_material=Co.Ms)
    sweep_results[d_nm] = (loop, m)
    print(f"   Hc={m.Hc*1e3:.1f} mT  Mr={m.Mr:.3f}  SQ={m.squareness:.3f}")


---
## 7. Summary plot


In [ ]:
fig = plt.figure(figsize=(13, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig)
ax_loops = fig.add_subplot(gs[0, :2])
ax_stats = fig.add_subplot(gs[0, 2])

colors = plt.cm.viridis(np.linspace(0.05, 0.95, len(diameters_nm)))
for (d_nm, (loop, met)), col in zip(sweep_results.items(), colors):
    ax_loops.plot(loop.B_applied*1e3, loop.Mx,
                  color=col, lw=1.8, label=f"d={d_nm:.0f} nm")

ax_loops.axhline(0, color="k", lw=0.6, ls="--")
ax_loops.axvline(0, color="k", lw=0.6, ls="--")
ax_loops.axvline( Co.mu0_Ha*1e3, color="orange", lw=1.2, ls=":",
                  label=f"SW Hc={Co.mu0_Ha*1e3:.0f} mT")
ax_loops.axvline(-Co.mu0_Ha*1e3, color="orange", lw=1.2, ls=":")
ax_loops.set_xlabel("mu0 H [mT]", fontsize=12)
ax_loops.set_ylabel("<Mx>/Ms", fontsize=12)
ax_loops.set_title("Hysteresis loops vs diameter", fontsize=11)
ax_loops.legend(fontsize=9, ncol=2)
ax_loops.grid(True, alpha=0.3)

d_arr  = np.array(diameters_nm)
Hc_arr = np.array([sweep_results[d][1].Hc * 1e3 for d in diameters_nm])
sq_arr = np.array([sweep_results[d][1].squareness for d in diameters_nm])

ax2 = ax_stats
ax2.plot(d_arr, Hc_arr, "o-", color="royalblue", lw=2, label="Hc [mT]")
ax2.axhline(Co.mu0_Ha*1e3, color="orange", lw=1.2, ls=":", label="SW limit")
ax3 = ax2.twinx()
ax3.plot(d_arr, sq_arr, "s--", color="firebrick", lw=1.8, label="Squareness")
ax2.set_xlabel("Diameter [nm]", fontsize=11)
ax2.set_ylabel("Hc [mT]", color="royalblue", fontsize=11)
ax3.set_ylabel("Squareness", color="firebrick", fontsize=11)
ax3.set_ylim(0, 1.1)
ax2.set_title("Hc & squareness vs d", fontsize=11)
ax2.grid(True, alpha=0.3)
lines1, labs1 = ax2.get_legend_handles_labels()
lines2, labs2 = ax3.get_legend_handles_labels()
ax2.legend(lines1+lines2, labs1+labs2, fontsize=8)

fig.suptitle("Co nanowires L=200 nm — diameter sweep",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("co_wire_diameter_sweep.png", dpi=150)
plt.show()
print("Saved: co_wire_diameter_sweep.png")

print()
print(f"  {'d [nm]':>8} {'Hc [mT]':>10} {'Hc/Ha':>8} {'Mr':>8} {'SQ':>8}")
print("-" * 50)
for d_nm in diameters_nm:
    met = sweep_results[d_nm][1]
    print(f"  {d_nm:8.0f} {met.Hc*1e3:10.1f} "
          f"{met.Hc/Co.mu0_Ha:8.3f} "
          f"{met.Mr:8.4f} {met.squareness:8.4f}")


---
## 8. Summary & next steps

### What we validated
- Newell tensor: OOMMF reference values, machine precision
- PBC demag: 4 analytical tests all pass
- Thin wire (d=4-10 nm): Hc approaches SW prediction (643 mT)

### Expected trends
- Small d (< ~18 nm) -> square loop, high squareness, Hc near SW
- Large d (> ~18 nm) -> Hc drops, loop less square (curling mode)
- d_crit ~ 3.6 l_ex ~ 18 nm for Co

### Next notebooks
1. `02_wire_array_PBC.ipynb` — Square/hex array, inter-wire coupling (PBC)
2. `03_material_comparison.ipynb` — Co vs FeCo vs Fe at same geometry
3. `04_parameter_sweep.ipynb` — BHmax optimisation vs d, pitch, L
